# Lab: Build a TOR-Like Server Node

In this lab, you will build server nodes that work together to create a TOR-like network. Each node will listen on a specific port, decrypt incoming packets, and forward them to the next node or send the actual request if it is the last node. The nodes will ensure secure and anonymous communication by encrypting and decrypting the data at each step.

### Objectives:

1. **Listening on a Specific Port**: Each node will listen on a designated port for incoming connections.

2. **Receiving and Decrypting Packets**: When a node receives a connection, it will receive the packet and decrypt its layer of encryption.

3. **Forwarding to the Next Node**: 
    - If the decrypted packet contains an IP address and port, the node will forward the remaining encrypted packet to the next node in the circuit.
    - The packet forwarded will still be encrypted (it will be the second layer of encryption).

4. **Sending the Actual Request**:
    - If the node is the last in the circuit, upon decryption, it will reveal the actual HTTP request.
    - The node will send the HTTP request to the target server and obtain the response.

5. **Returning the Response**:
    - The node will return the response to the parent node, encrypting it with the key to maintain the security and anonymity of the communication. The response must follow the circuit until it gets to the client.

### Steps:

1. **Listening on a Specific Port**:
    - Set up each node to listen on a designated port for incoming connections.

2. **Receiving and Decrypting Packets**:
    - When a node receives a packet, it will decrypt its layer using its private key.

3. **Forwarding to the Next Node**:
    - If the decrypted packet contains an IP address and port, the node will forward the remaining encrypted packet to the next node in the circuit.
    - Ensure the packet remains encrypted for the next node.

4. **Sending the Actual Request**:
    - If the node is the last in the circuit, it will decrypt the packet to reveal the HTTP request.
    - Send the HTTP request to the target server and obtain the response.

5. **Returning the Response**:
    - Encrypt the response.
    - Send the encrypted response back through the circuit to the client.

### Tips:

Watchout with the lenght of the packets. Most encryption errors could be due this, so you'll maybe have to send and handle chunks. Every time the packet is encrypted, it's size will change

In [ ]:
#Note: This is how I did it. It's just one way, there are multiple ways on doing this.

import socket
import threading
import os
import ssl
import base64
from time import sleep
from cryptography.fernet import Fernet


# Function to encrypt a message using the session key
def encrypt_message(message, session_key):
    f = Fernet(session_key)
    # Fernet works with bytes - convert if needed
    if isinstance(message, str):
        message = message.encode()
    encrypted = f.encrypt(message)
    return encrypted

# Function to decrypt a message using the session key
def decrypt_message(encrypted_message, session_key):
    f = Fernet(session_key)
    decrypted = f.decrypt(encrypted_message)
    return decrypted


class Node:
    PORT_START = 5000
    def __init__(self, id, prev_node=None, next_node=None, session_key=None):
        self.id = id
        self.port = self.PORT_START + id
        self.prev_node:Node = prev_node
        self.next_node:Node = next_node
        self.session_key = session_key


    def handle_client(self, conn:socket.socket, addr):
        try:
            sleep(1)  # Wait for the client to send data
            conn.settimeout(1.0)  # Set a timeout for receiving data
            tf = True
            decrypted_data = None
            while True:
                try:
                    data = conn.recv(4098)
                    if not data:
                        break
                    
                    # Decrypt the incoming data with this node's session key
                    decrypted_data = decrypt_message(data, self.session_key)
                    
                except socket.timeout:
                    break
                except Exception as e:
                    print(f"Node {self.id} Error during receiving: {e}")
                    break
            
            if nodeNum > 0:
                # Forwarding node - send decrypted data to next node
                # The decrypted data still contains more layers of encryption
                self.send_data(decrypted_data)
            else:
                # Exit node - this is the last node in the circuit
                # Create a SSL Socket for HTTPS connection
                dest_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                context = ssl.create_default_context(ssl.Purpose.SERVER_AUTH)
                
                # Get the final host from the HTTP Header
                address = self.extract_host(decrypted_data)
                
                with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as dest_socket:
                    with context.wrap_socket(dest_socket, server_hostname=address) as ssock:
                        # Connect to the final host
                        ssock.connect((address, 443))
                        # Send the decrypted data to the final host
                        ssock.sendall(decrypted_data)
                        
                        # Receive the response from the final host
                        # May need multiple recv calls for large responses
                        response = b""
                        while True:
                            chunk = ssock.recv(4096)
                            if not chunk:
                                break
                            response += chunk
                        
                        # Encrypt the response with this node's session key
                        encrypted_response = encrypt_message(response, self.session_key)
                        
                        # Send the encrypted response back through the circuit
                        self.send_data(encrypted_response, conn)
                        

        except Exception as e:
            print(f"Node {self.id} Error: {e}")
        finally:
            conn.close()
    
    # This function extracts the host from the HTTP Header
    def extract_host(self, request_bytes):
        request_str = request_bytes.decode('utf-8')  # Decode the bytes to a string
        lines = request_str.split('\r\n')  # Split the request into lines
        for line in lines:
            if line.startswith('Host: '):
                host = line.split(' ')[1]  # Extract the host part
                return host
        return None
    
    def send_data(self, data, s=None):
        """Send data to the next node or back through the circuit.
        
        Args:
            data: The data to send (bytes)
            s: If provided, send back through this existing socket (to previous node).
               If None, create new connection to next node.
        """
        if s:
            # Send back through existing socket (to previous node in circuit)
            s.sendall(data)
        else:
            # Create new connection to next node
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
                sock.connect(('127.0.0.1', self.next_node.port))
                sock.sendall(data)

    def start(self):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(('127.0.0.1', self.port))
            s.listen()
            print(f"Node {self.id} listening on port {self.port}")
            while True:
                conn, addr = s.accept()
                client_thread = threading.Thread(target=self.handle_client, args=(conn, addr))
                client_thread.start()

In [ ]:
# Client code - Creating an onion-encrypted request

import socket

# Sample HTTP request
http_request = b"""GET / HTTP/1.1\r
Host: example.com\r
Connection: close\r
\r
"""

# Onion encryption: Encrypt in reverse order (last layer first)
# Layer 3 (Node 2 - exit node): Encrypt with key3
layer3 = encrypt_message(http_request, key3)

# Layer 2 (Node 1): Encrypt layer3 with key2
layer2 = encrypt_message(layer3, key2)

# Layer 1 (Node 0): Encrypt layer2 with key1
onion_packet = encrypt_message(layer2, key1)

print(f"Original HTTP request size: {len(http_request)} bytes")
print(f"After 3 layers of encryption: {len(onion_packet)} bytes")

# Send the onion packet to the first node in the circuit
def send_through_circuit(onion_packet, first_node_port=5002):
    """Send onion packet through the TOR-like circuit"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect(('127.0.0.1', first_node_port))
        sock.sendall(onion_packet)
        
        # Wait for response to come back through the circuit
        response = sock.recv(8192)
        return response

# To test: First start the nodes with node.start() in separate terminals/notebooks
# Then uncomment and run:
# response = send_through_circuit(onion_packet)
# print(f"Response size: {len(response)} bytes")

In [ ]:
# Example usage - Create a 3-node circuit

# Generate unique session keys for each node
key1 = Fernet.generate_key()
key2 = Fernet.generate_key()
key3 = Fernet.generate_key()

# Create nodes in a chain
# Node 0 (port 5000) -> Node 1 (port 5001) -> Node 2 (port 5002) [exit node]
node0 = Node(id=0, session_key=key1)
node1 = Node(id=1, next_node=node0, session_key=key2)  # Will forward to node0
node2 = Node(id=2, next_node=node1, session_key=key3)  # Will forward to node1

# For testing, set nodeNum to indicate which node behavior to test
# nodeNum > 0: forwarding node (sends to next node)
# nodeNum <= 0: exit node (sends actual HTTP request)

print("Nodes created successfully!")
print(f"Node 0: port {node0.port}, key: {key1.decode()[:10]}...")
print(f"Node 1: port {node1.port}, key: {key2.decode()[:10]}...")
print(f"Node 2: port {node2.port}, key: {key3.decode()[:10]}...")